# Class 17: Cars 2 Has an Alibi
### Exploratory Data Analysis and Correlation

Twenty-one Pixar films, three rating services. **Record your answers on the paper handout, not here.**

In [ ]:
import numpy as np
from datascience import *
import matplotlib.pyplot as plt
%matplotlib inline

print("Ready!")

---
## Part 1. One Variable at a Time

In [ ]:
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-03-11/public_response.csv'
ratings = Table.read_table(url)
# If the network is unavailable, use the local copy instead:
# ratings = Table.read_table('data/public_response.csv')
ratings.show(3)

Some rows have `nan` markers for missing data. `Table` has no built-in way to drop those, but pandas DataFrames do — so convert, drop, and convert back.

**Remember this trick.** It will come in handy for the final project.

In [ ]:
df = ratings.to_df().dropna()
ratings = Table().from_df(df)

rmc = ratings.select("rotten_tomatoes", "metacritic", "critics_choice")
print(f"{rmc.num_rows} films with complete numeric ratings")
rmc.show(3)

### Method 1: Look at the data

For a small table, just reading it is a legitimate first step.

In [ ]:
ratings.select("film", "rotten_tomatoes", "metacritic", "critics_choice").show()

### Method 2: Summary statistics

In [ ]:
rmc.stats(ops=[np.min, np.max, np.ptp, np.mean, np.median, np.std])

> **Handout Q1.1 and Q1.2**

### Method 3: Box plots

Do **Q1.3 on paper first** — work out the quartiles by hand before you let matplotlib draw them.

In [ ]:
# Quartiles for each rating service
for col in rmc.labels:
    values = rmc.column(col)
    q1, q2, q3 = percentile(25, values), percentile(50, values), percentile(75, values)
    print(f"{col:18s} Q1={q1:5.1f}  median={q2:5.1f}  Q3={q3:5.1f}  IQR={q3-q1:5.1f}")

In [ ]:
rmc.boxplot()
ax = plt.gca()
ax.set_title("Score Comparison")
ax.set_ylabel("Rating")
plt.show()

> **Handout Q1.4 and Q1.5**

---
## Part 2. Two Variables at a Time

*Bivariate* means looking at two variables together. A scatter plot is the place to start.

In [ ]:
rmc.scatter("metacritic", "critics_choice")

> **Handout Q2.1 — commit to an answer before computing anything.**

### Standard units

Subtract the mean, divide by the standard deviation. The result has mean 0 and SD 1.

In [ ]:
def standard_units(any_numbers):
    "Convert any array of numbers to standard units."
    return ...

# Test it
x = make_array(1, 3, 2, 8, 3, 4, 11)
y = standard_units(x)
print(f"The mean of y is {np.mean(y):.3f}")
print(f"The standard deviation of y is {np.std(y):.3f}")

In [ ]:
for col in rmc.labels:
    rmc = rmc.with_columns(col + "_su", standard_units(rmc.column(col)))
rmc.show(3)

In [ ]:
rmc.scatter("metacritic_su", "critics_choice_su")

> **Handout Q2.2 and Q2.3**

### The correlation coefficient

$r$ is the average of the product of the two variables in standard units.

In [ ]:
def correlation(x, y):
    "Correlation between two arrays."
    return np.mean(...)

r = correlation(rmc.column("metacritic"), rmc.column("critics_choice"))
print(f"metacritic vs critics_choice:  r = {r:.3f}")

In [ ]:
pairs = [("rotten_tomatoes", "metacritic"),
         ("rotten_tomatoes", "critics_choice"),
         ("metacritic", "critics_choice")]

for a, b in pairs:
    print(f"{a:18s} vs {b:18s}  r = {correlation(rmc.column(a), rmc.column(b)):.3f}")

> **Handout Q2.4**

---
## Part 3. The Alibi

One film stood out on the box plot and stands out on the scatter plot. Test whether it is actually the point distorting your correlation.

In [ ]:
films = ratings.column("film")
rt    = ratings.column("rotten_tomatoes")
mc    = ratings.column("metacritic")

keep = films != "Cars 2"

print(f"r with all {len(rt)} films:        {correlation(rt, mc):.3f}")
print(f"r with Cars 2 removed (n={keep.sum()}):  {correlation(rt[keep], mc[keep]):.3f}")

> **Handout Q3.1**

### Leave each film out, one at a time

In [ ]:
base = correlation(rt, mc)

changes = []
for film in films:
    others = films != film
    changes.append(correlation(rt[others], mc[others]) - base)

influence = Table().with_columns("film", films, "change in r", np.round(changes, 4))
influence.sort("change in r", descending=True).show(5)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(rt, mc, color="steelblue", s=40)
for film in ["Cars 2", "Onward"]:
    i = np.where(films == film)[0][0]
    plt.annotate(film, (rt[i], mc[i]), textcoords="offset points", xytext=(8, -4))
plt.xlabel("rotten_tomatoes"); plt.ylabel("metacritic")
plt.title(f"r = {base:.3f}")
plt.show()

> **Handout Q3.2**

### The hypothetical

Cars 2 scored 57 on metacritic. Given how the other low-rated films did, a score near **46** would have been more in keeping with the pattern. What if it had scored 46?

In [ ]:
i = np.where(films == "Cars 2")[0][0]

mc_alibi = mc.copy()
mc_alibi[i] = 46

print(f"r as observed:                  {correlation(rt, mc):.3f}")
print(f"r if Cars 2 had scored 46:      {correlation(rt, mc_alibi):.3f}")
print(f"r with Cars 2 removed:          {correlation(rt[keep], mc[keep]):.3f}")

In [ ]:
print(f"rotten_tomatoes WITH Cars 2:     range {rt.max() - rt.min():.0f},  SD {np.std(rt):.2f}")
print(f"rotten_tomatoes WITHOUT Cars 2:  range {rt[keep].max() - rt[keep].min():.0f},  SD {np.std(rt[keep]):.2f}")

> **Handout Q3.3 and Q3.4**

### Four data sets

These four share the same means, the same standard deviations, and the same correlation.

In [ ]:
x_a = make_array(10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5)
y_1 = make_array(8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68)
y_2 = make_array(9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74)
y_3 = make_array(7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73)
x_d = make_array(8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8)
y_4 = make_array(6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89)

sets = [("I", x_a, y_1), ("II", x_a, y_2), ("III", x_a, y_3), ("IV", x_d, y_4)]

for name, x, y in sets:
    print(f"Set {name:3s}  mean x = {np.mean(x):.2f}   mean y = {np.mean(y):.2f}   "
          f"SD y = {np.std(y):.2f}   r = {correlation(x, y):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, x, y) in zip(axes, sets):
    ax.scatter(x, y, color="steelblue", s=40)
    ax.set_title(f"Set {name}")
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)
plt.tight_layout()
plt.show()

> **Handout Q3.5**